In [2]:
# missing values
from pyexpat import model
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np


In [3]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(f"Duplicate values of dataset: {df.duplicated().sum()}")
print(f"Missing value of dataset: {df.isnull().sum()}")

Duplicate values of dataset: 0
Missing value of dataset: MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [42]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [14]:
# preprocessing
def load_data():
    """
    Returns:
    The California housing data as a DataFrame.
    """  
    return fetch_california_housing(as_frame=True).frame
# df = load_data()
# print(df.info())
def encode_categoricals(df, columns):
    """One-hot encode categorical columns."""
    df = df.copy()
    df = pd.get_dummies(df, columns=columns, drop_first=True, dtype=int)
    return df
# df = encode_categoricals(load_data(),['HouseAge'])
# print(df.head())
def check_missing_values(df):
    """
    Check for missing values in the DataFrame.
    Returns:
    A boolean indicating if there are missing values.
    """
    if df is None:
        raise ValueError("DataFrame is None. Please provide a valid DataFrame.")
    
    return df.isnull().values.any()
def check_duplicates(df):
    """
    Check for duplicate rows in the DataFrame.
    Returns:
    A boolean indicating if there are duplicate rows.
    """
    if df is None:
        raise ValueError("DataFrame is None. Please provide a valid DataFrame.")
    
    return df.duplicated().any()

def splitting_data(df):
    """
    Split the input DataFrame into training and test data.
    Returns:
    Training and validation features and targets.
    """
    if df is None:
        raise ValueError("DataFrame is None. Please provide a valid DataFrame.")
    
    target = df['MedHouseVal']
    features = df.drop('MedHouseVal', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(
        features, target, test_size=0.25, random_state=42
    )

    return x_train, x_test, y_train, y_test
def scale_features(x_train, x_test):
    """
    Scale the features using StandardScaler.
    Returns:
    Scaled training and validation features.
    """
    if x_train is None or x_test is None:
        raise ValueError("Training or test features are None. Please provide valid inputs.")
    
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    return x_train_scaled, x_test_scaled
# df = load_data()
# df_scaled = encode_categoricals(df, ['HouseAge'])
# print(f"Feature scaling completed. Scaled DataFrame shape: {df_scaled.shape}")
def feature_target_corr():
    """
    Calculate the correlation between features and the target variable.
    Returns:
    A DataFrame containing features and their correlation with the target.
    """
    df = load_data()
    if df is None:
        raise ValueError("DataFrame is None. Please provide a valid DataFrame.")
    
    corr_matrix = df.corr()
    target_corr = corr_matrix['MedHouseVal'].drop('MedHouseVal')
    importance_df = target_corr[target_corr > 0.1].sort_values(ascending=False)
    return importance_df

In [15]:

def model_training(df):
    """
    Train a Random Forest Regressor on the training data.
    Returns:
    The trained model.
    """
    # df_split = df.copy()
    x_train,x_test, y_train,y_test = splitting_data(df)
    rf = RandomForestRegressor()
    rf_fit = rf.fit(x_train, y_train)
    return rf_fit
model = model_training(df)
print(model)


RandomForestRegressor()


In [16]:
def test_model():
    """
    Test the model_evaluation function with a simple linear regression model.
    """
    x_train,x_test, y_train,y_test = splitting_data(df)
    model = model_training(df)
    predictions = model.predict(x_test)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    assert mse >= 0
    assert rmse >= 0
    return mse, rmse
df_test = test_model()
print(df_test)

(0.2543638451317228, 0.5043449663987168)


In [18]:
def feature_importance(rf, feature_names):
    """
    Calculate feature importance from the fitted Random Forest model.
    Returns:
    DataFrame containing features and their importance scores.
    """
    if rf is None or feature_names is None:
        raise ValueError("Model or feature names are None. Please provide valid inputs.")
    # df = load_data()
    # rf = model_training()
    # rf = rf_model(df)
    importances_val = rf.feature_importances_

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances_val,
        'importance_Percent': importances_val * 100
    }).sort_values('importance', ascending=False)

    return importance_df
# df_feat = model_training(df)
# ft = feature_importance(df_feat, x_train.columns)
# print(ft)

In [20]:
def test_feature_imp(rf, feature_names):
    """Test feature importance function works."""
    feat_imp = feature_importance(rf, feature_names)
    
    # assert rf.empty, "<Data frame is empty.>"
    assert len(feature_names) > 0, "The feature names empty."
    assert not feat_imp.empty, "< The dataset is invalid.>"
    return feat_imp
# df = load_data()
# df_test_feat_imp = model_training(df)
# test = test_feature_imp(df_test_feat_imp, x_train.columns)
# print(test) 

In [ ]:
def model_evaluation(model, x_test, y_test):
    """
    Evaluate the performance of a Random Forest Regressor on the test data.
    Returns:
    The trained model and its performance metrics.
    """
    predictions = model.predict(x_test)
    mse_rf_tuned = mean_squared_error(y_test, predictions)
    rmse_rf_tuned = np.sqrt(mse_rf_tuned)
    r2_rf_tuned = r2_score(y_test, predictions)
    print("Metrics:")
    # print(f"predictions: {predictions}")
    print(f"MSE:  {mse_rf_tuned:.4f}")
    print(f"RMSE: {rmse_rf_tuned:.4f}")
    print(f"R²:   {r2_rf_tuned:.1%}")
    print()
    return mse_rf_tuned, rmse_rf_tuned, r2_rf_tuned,predictions
df = load_data()
model_eva= model_training(df)
x_train, x_test, y_train, y_test = splitting_data(df)

df_model_evaluation = model_evaluation(model_eva, x_test, y_test)

Metrics:
MSE:  0.2514
RMSE: 0.5014
R²:   81.0%



In [ ]:
def test_model_evaluation(df):
    x_train, x_test, y_train, y_test = splitting_data(df)
    trained_model = model_training(df)
    test_model_eval = model_evaluation(model, x_test, y_test)
    mse, rmse, r2, predictions = model_evaluation(
        trained_model, x_test, y_test
    )
    assert mse >= 0, f"Negative MSE"
    assert r2 <= 50.0, f" R2 score must be more than 50%"
    # assert np.isfinite(r2), "R² must be finite."

    assert len(predictions) == len(y_test)
    return mse, r2

model_eval_test = test_model_evaluation(df)


Metrics:
MSE:  0.2549
RMSE: 0.5049
R²:   80.7%

Metrics:
MSE:  0.2535
RMSE: 0.5035
R²:   80.8%



## Model Test


In [22]:
np.random.seed(42)
n_rows = 200

df_sample = pd.DataFrame({
    "size_sqft": np.random.randint(500, 3500, n_rows),
    "num_rooms": np.random.randint(1, 6, n_rows),
    "age_years": np.random.randint(0, 50, n_rows),
})

# Target: house price (float), based on features + noise
df_sample["MedHouseVal"] = (
    df_sample["size_sqft"] * 150
    + df_sample["num_rooms"] * 5000
    - df_sample["age_years"] * 300
    + np.random.normal(0, 10000, n_rows)
).round(2)  # keep target as float

# # ---- 2. Split features (X) and target (y) ----
# X = df_sample[["size_sqft", "num_rooms", "age_years"]]
# y = df_sample["price"]

In [23]:
sample_splitt_data = train_test_split(df_sample)

In [24]:
splitting_data_sample = splitting_data(df_sample)
splitting_data_sample

(     size_sqft  num_rooms  age_years
 114       2682          5         35
 173       2029          3         38
 5         2669          5         25
 126       3273          1         35
 117       1279          3         35
 ..         ...        ...        ...
 106        891          3         33
 14        1269          1         33
 92         879          1         11
 179       1806          1          4
 102       1995          3         28
 
 [150 rows x 3 columns],
      size_sqft  num_rooms  age_years
 95        2562          5          1
 15        2891          1         31
 30        3058          1         36
 158       1683          2          6
 128       1460          5         48
 115        700          3         24
 69        2000          1         48
 170       2481          4         35
 174       2538          1         44
 45        2714          1         47
 66        1521          5         23
 182       1472          3         28
 165       3193        

In [25]:
sample_model = model_training(df_sample)
sample_model

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [30]:
# def model_evaluation(model, x_test, y_test):
x_train, x_test, y_train, y_test = splitting_data(df_sample)
sample_model_eval = model_evaluation(sample_model, x_test, y_test)
sample_model_eval

Metrics:
MSE:  204978276.1973
RMSE: 14317.0624
R²:   98.4%



(204978276.19726068,
 14317.062415078753,
 0.9839667925300171,
 array([394917.6146, 431719.1003, 461576.6786, 270510.7872, 232880.8918,
        124520.6261, 296742.7273, 390692.2688, 383264.2218, 412733.7763,
        234683.1206, 233490.5976, 490264.3278, 428518.9572, 303708.4791,
        297943.8517, 286020.2543, 423360.734 , 234222.4681, 251386.024 ,
        486150.8345, 303240.4723, 101821.0955, 145135.7465, 325519.3859,
        469033.2776, 496085.3942, 473701.6579, 358355.3098, 234835.1637,
        310968.5581, 452182.9008, 512743.3843, 121629.0662, 331399.3613,
        315294.7283, 149739.5712, 504800.9278, 325630.8664, 387247.293 ,
        201601.8917, 354674.3985, 358556.2294,  90219.5366, 204723.9926,
        316138.1136, 245645.8106, 273214.2642, 316050.1778, 200060.6374]))